In [1]:
import os
import chromadb
import dotenv
from agents import Agent, Runner, WebSearchTool, SQLiteSession, function_tool, trace
from agents.mcp import MCPServerStreamableHttp

dotenv.load_dotenv()

True

### Get existing database

In [ ]:
chroma_client = chromadb.PersistentClient(path="../chroma")
collection_name = chroma_client.list_collections()[0].name
database = chroma_client.get_collection(name=collection_name)

### Retrieve calorie information from the database based on user query

In [ ]:
@function_tool
def get_calorie_info(query: str, top_k: int = 3) -> str:
    """
    Based on user's query, retrieve relevant calorie information from the RAG database.

    Args:
        query (str): The food item to search for.
        top_k (int): The maximum number of top relevant documents to retrieve.

    Returns:
        str: A formatted string containing the retrieved calorie information of the queried food item.
    """

    output = database.query(
        query_texts=[query],
        n_results=top_k
    )

    if not output["documents"][0]:
        return "No relevant nutrition information found for: {query}."
    
    formatted_output = []
    for id, metadata in enumerate(output['metadatas'][0]):
        food_item = metadata['food_item']
        calories = metadata['calories_per_100g']
        category = metadata['food_category']

        formatted_output.append(f"{food_item} ({category}): {calories} calories per 100g")

    return "Nutrition information:\n" + "\n".join(formatted_output)

In [5]:
# Test retrieving results (comment the line @function_tool in get_calorie_info to run this)
output = get_calorie_info("apple")
print(output)

Nutrition information:
Apple (Fruits): 52 calories per 100g
Apple Spritzer (Non-AlcoholicDrinks&Beverages): 24 calories per 100g
Apple Juice ((Fruit)Juices): 46 calories per 100g


### Connect to Exa MCP Server for web search

In [ ]:
exa_mcp = MCPServerStreamableHttp(
    name="Exa Search",
    params={
        "url": f"https://mcp.exa.ai/mcp?{os.getenv('EXA_API_KEY')}",
        "timeout": 30,
    },
    client_session_timeout_seconds=30,
    cache_tools_list=True,
    max_retry_attempts=1,
)

await exa_mcp.connect()

### Create the first Agent

In [8]:
# Make sure to uncomment the line @function_tool in get_calorie_info to use it as a tool
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    * You are a helpful nutrition assistant that provides concise and accurate calorie information.
    * You follow this workflow strictly:
        1) Use the get_calorie_info to find calorie data for the requested food. Only use the result if it exactly matches the query.
        2) If no exact match or ingredient details are needed, use Exa Search to find the precise recipe and ingredient list.
        Even if the search provides calorie data, always re-fetch ingredient calories using the get_calorie_info for consistency.
        3) After identifying the full ingredient list, use the get_calorie_info to get calorie information for each ingredient.
        4) For meal-related queries, output a list of ingredients with quantities and calories per single serving, and include the total calories.
        5) Don't use the get_calorie_info more than 10 times.
        6) Always be concise and factual in your answers.
    """,
    tools=[get_calorie_info],
    mcp_servers=[exa_mcp],
)

### Create the second Agent

In [9]:
planner_agent = Agent(
    name="Meal Planner Assistant",
    instructions="""
    * You are a helpful assistant that suggests healthy breakfast options.
    * You provide concise answers tailored to the user's preferences.
    * Based on the user's input, suggest several healthy breakfast meals that are suitable for a busy person.
    * For each meal, explicitly mention its name and include one sentence explaining why it's a healthy choice.
    """,
)

### Create the third Agent for handoff

In [10]:
price_checker_agent = Agent(
    name="Price Checker Assistant",
    instructions="""
    * You are a helpful assistant that analyzes multiple breakfast meals. Each meal includes its ingredients and calorie values.
    * Your task is to find the approximate price of each ingredient and summarize the results clearly following this workflow:

    1) Use the web search tool to find approximate prices for each ingredient.
    2) In your final output, include the meal name and a list of ingredients with their calories and estimated prices.
    3) Format your response in concise Markdown for easy readability. 
    """,
    tools=[WebSearchTool()],
)

### Create the fourth Agent using Agents 1&2 as tools and Agent 3 for handoff

In [11]:
calorie_calculator = calorie_agent.as_tool(
    tool_name="calorie_calculator",
    tool_description="Use this tool to calculate the calories of a meal and its ingredients",
)

meal_planner = planner_agent.as_tool(
    tool_name="meal_planner",
    tool_description="Use this tool to generate personalized healthy breakfast options based on dietary preferences and requirements.",
)

meal_advisor = Agent(
    name="Meal Advisor",
    instructions="""
    * You are a Meal Advisor who creates personalized, healthy breakfast meal plans based on user preferences.
    * For each meal, you must include the meal name, a list of ingredients, and the calorie count for both the meal and its ingredients.

    Follow this workflow carefully:
    1) Use the meal_planner tool to generate several healthy breakfast options tailored to the user's preferences.
    2) Use the calorie_calculator tool to determine the calorie content of each meal and its ingredients.
    3) Once the meal plans and calorie data are ready, handoff this information to the Price Checker Assistant to include the corresponding prices.

    """,
    tools=[meal_planner, calorie_calculator],
    handoff_description="""
    Create a concise breakfast recommendation based on the user's preferences. Use Markdown format.
    """,
    handoffs=[price_checker_agent],
)

### Create short-term memory and run agents

In [12]:
session = SQLiteSession("conversation_history")
result = await Runner.run(
    meal_advisor,
    "I'm a busy person who wants healthy, easy-to-make breakfasts. I like oatmeal and eggs. Please suggest two breakfast options, including their ingredients and calorie counts.",
    session=session
)
print(result.final_output)

Here are two easy, healthy breakfast options with ingredients, approximate calories, and estimated prices.

1) Savory Oatmeal Bowl with Fried Egg
- Calories: ~350–380 kcal
- Ingredients and estimated prices
  - 1/2 cup rolled oats: ~$0.13
  - 1 cup water or broth: $0–$0.50 (broth adds cost)
  - handful spinach (about 1 cup): ~$0.21
  - 1 egg: ~$0.49
  - 1 tbsp shredded cheese: ~$0.20
  - salt, pepper, hot sauce: negligible
  - optional: small tomato or avocado for freshness: ~$0.50–$1.50
- Estimated total: ~$1.53 (base) to ~$3.03 (with tomato/avocado)
- Notes: Quick microwavable; portable and balanced macros.

2) Egg-Oat Muffin Cups (make-ahead)
- Calories: ~158 kcal per muffin (batch total ~950 kcal for 6 muffins)
- Ingredients and estimated prices
  - 6 large eggs: ~$0.49 each = ~$2.94
  - 1 cup rolled oats: ~$0.27
  - 1/2 cup shredded cheese: ~$0.78
  - 1 cup chopped veggies: ~$0.50
  - salt/pepper: negligible
- Estimated total for batch: ~$4.72; per muffin: ~$0.79
- Notes: Bake onc

### Check short-term memory

In [13]:
with trace("Multi Agent: Meal Advisor"):
    result = await Runner.run(
        meal_advisor,
        "Help me modify the options without using cheese.",
        session=session
    )
    print(result.final_output)

Here are the two options updated to exclude cheese.

1) Savory Oatmeal Bowl with Fried Egg (no cheese)
- Calories: ~300–330 kcal (base); ~360–390 kcal if you add avocado or tomato
- Ingredients
  - 1/2 cup rolled oats
  - 1 cup water or low-sodium broth
  - handful spinach (about 1 cup)
  - 1 egg
  - salt, pepper, hot sauce to taste
  - optional: small tomato or avocado for freshness
- Method
  - Microwave oats with water/broth 2–3 minutes.
  - Stir in spinach until wilted; season.
  - Fry or poach the egg; place on top.

2) Egg-Oat Muffin Cups (make-ahead, no cheese)
- Calories: ~129 kcal per muffin (batch ~770 kcal for 6 muffins)
- Ingredients
  - 6 large eggs
  - 1 cup rolled oats (or oat flour)
  - 1 cup chopped veggies (e.g., bell peppers, spinach, mushrooms)
  - salt/pepper
  - optional: herbs/spices for flavor
- Method
  - Preheat oven to 375°F (190°C). Mix eggs, oats, veggies; season.
  - Pour into a greased muffin tin; bake 18–20 minutes until set. Cool and refrigerate.
  - Re